In [0]:
#Imports to run API Data Scraping script

import requests
import logging
import traceback
from datetime import datetime
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
#parameters that need to be changed between notebooks

schema_path = 'fake_general_ledger.default'
volume_path = '/Volumes/fake_general_ledger/default/territory'
volume_identifier = 'Territory'
table_name = 'bronze_gl_territory'
spark = SparkSession.builder.appName("bronze_level_ingestion").getOrCreate()

In [0]:
#Set up logger to track pipeline activities

def setup_logging():

    # Ensure logs directory exists before starting logging
    os.makedirs('logs', exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'logs/bronze_level_territory_run_{datetime.now().strftime('%Y%m%d')}log'),logging.StreamHandler(sys.stdout)]
    )

    return logging.getLogger(__name__)

In [0]:
# ---------------------------------------------------------
# 1. FILE DISCOVERY
# ---------------------------------------------------------

def get_correct_files(logger, volume_path, volume_identifier):

    volume = []

    for vol_path, _, filenames in os.walk(volume_path):

        for file in filenames:

            if volume_identifier in file:

                full_path = os.path.join(vol_path, file)

                logger.info(f'Found file: {full_path}')

                volume.append(full_path)

    logger.info(f'Total files discovered: {len(volume)}')

    return volume

In [0]:
# ---------------------------------------------------------
# 2. READ FILE + APPLY METADATA
# ---------------------------------------------------------

def prepare_bronze_dataframe(logger, file_path):

    logger.info(f"Preparing dataframe for: {file_path}")

    # ---------------------------------------------
    # READ SOURCE FILE
    # ---------------------------------------------

    df = spark.read.option("header", True).option("inferSchema",True).csv(file_path)

    # ---------------------------------------------
    # APPLY METADATA COLUMNS
    # ---------------------------------------------

    df_with_metadata = df.withColumns({
            "_bronze_ingestion_timestamp": F.current_timestamp(),
            "_bronze_source_file": F.lit(file_path),
            "_bronze_file_name": F.element_at(F.split(F.lit(file_path),"/"), -1),
            "_bronze_load_date": F.current_date(),
            "_bronze_pipeline_run_id": F.expr("uuid()")
            })

    logger.info(f"Metadata columns applied to: {file_path}")

    return df_with_metadata

In [0]:
# ---------------------------------------------------------
# 3. APPEND DATAFRAME TO BRONZE TABLE
# ---------------------------------------------------------

def create_or_append_to_bronze_table(logger, df, table_name):

    logger.info(f"Appending dataframe to: {table_name}")

    df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(f'{schema_path}.{table_name}')

    logger.info(f"Successfully appended to: {table_name} in {schema_path}")



In [0]:
# ---------------------------------------------------------
# 4. DRIVER CODE
# ---------------------------------------------------------

def bronze_level_ingestion(schema_path, table_name, volume_path, volume_identifier):

    logger = setup_logging()
    spark = SparkSession.builder.appName("bronze_level_ingestion").getOrCreate()

    files = get_correct_files(logger, volume_path, volume_identifier)

    for file_path in files:

        df = prepare_bronze_dataframe(logger, file_path=file_path)

        create_or_append_to_bronze_table(logger, df, table_name)
        
        logger.info(f'Bronze level table {table_name} created or appended to successfully.')

In [0]:
bronze_level_ingestion(schema_path, table_name, volume_path, volume_identifier)

In [0]:
%sql
SELECT * FROM fake_general_ledger.default.bronze_gl_territory

In [0]:
%sql

DROP TABLE IF EXISTS fake_general_ledger.default.bronze_gl_territory 